# P4 실습: Transformer와 LLM

이 실습의 목표는 **모델을 훈련하는 것이 아니라 사전훈련된 모델을 사용하여 Transformer의 작동 원리를 관찰**하는 것이다.

**네 단계**
1. 텍스트 → 토큰 → 토큰 ID → 임베딩
2. 간단한 Q·K·V와 Self-Attention 관찰
3. **Encoder–Decoder**로 영어→스페인어 번역
4. **Decoder-only**로 다음 토큰 예측, 텍스트 이어쓰기·질의응답

전체 실습에는 **훈련·파인튜닝·역전파 코드가 없다.** 모두 실행 가능한 완성형 코드이며, 학생은 문장과 설정을 바꿔 결과를 비교한다. **의미 검색과 RAG의 실제 구현은 P4 Project에서 수행한다.**


## 실행 환경

- VS Code/Jupyter Notebook 또는 Google Colab에서 실행할 수 있다.
- 처음 실행하면 Hugging Face에서 **두 개의 사전훈련 모델**을 다운로드한다. 네트워크 속도에 따라 시간이 걸린다.
- PyTorch가 없는 환경이라면 별도로 `%pip install torch`를 실행한다. CPU만으로도 가능하지만 GPU보다 느릴 수 있다.
- 설치 직후 import 오류가 발생하면 **커널을 재시작한 후 아래 셀부터** 실행한다.

**사용 모델:** `Helsinki-NLP/opus-mt-en-es`(영어→스페인어, Encoder–Decoder 계열), `HuggingFaceTB/SmolLM2-135M-Instruct`(영어 중심의 소형 Decoder-only 모델). 최신 대형 대화형 모델 수준의 정확성을 기대하지 않는다.


In [ ]:
%pip install -q "transformers>=4.47,<5" sentencepiece sacremoses


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

# P4에서는 학습하지 않고 사전훈련 모델로 추론만 한다.
print("PyTorch:", torch.__version__)


---
## 텍스트를 숫자로 표현하는 과정 확인

이 실습에서는 **하나의 문장** `The unbelievable red car is fast.`를 사용해 표준화, 토큰화, 어휘집, 임베딩과 위치 인코딩을 살펴본다.

먼저 모델 다운로드가 필요 없는 **교육용 예제**에서 단어 기준 토큰 7개, 수업용 ID 10~16, 임베딩 차원 256을 사용한다. 이후 **이미 학습된 번역 모델의 실제 토크나이저와 가중치**를 확인한다. 실습 내내 모델은 **직접 훈련하지 않는다.**


### 표준화: 띄어쓰기 정리하기

**표준화**(Text Standardization)는 텍스트를 정해진 표기 규칙에 따라 일관되게 정리하는 것이다. 같은 공통 문장에 들어간 앞뒤 공백과 연속된 공백을 제거한다. **대소문자와 마침표는 유지**하므로 다음 단계에서도 항상 같은 문장을 사용한다.


In [ ]:
import re

raw_text = "  The   unbelievable  red car is fast.  "
example_text = re.sub(r"\s+", " ", raw_text.strip())

print("표준화 전:", repr(raw_text))
print("표준화 후:", repr(example_text))
assert example_text == "The unbelievable red car is fast."

### 토큰화: 단어와 부분단어 비교

**토큰화**(Tokenization)는 문장을 토큰으로 나누는 과정이다. 이번 코드에서는 같은 문장을 **단어 기준**과 **부분단어 기준**으로 각각 나누어 결과를 비교한다. 부분단어의 `un` / `believ` / `able`은 설명을 위한 **가상 분할**이며, 실제 번역 모델의 토큰화 결과는 뒤에서 확인한다.

이후 교육용 ID와 임베딩에서는 **단어 기준 7개 토큰만 사용**한다.


In [ ]:
word_tokens = re.findall(r"[A-Za-z]+|[.!?]", example_text)
subword_demo = ["The", "un", "believ", "able", "red", "car", "is", "fast", "."]

print("같은 문장:", example_text)
print("단어 기준:", word_tokens, "→", len(word_tokens), "개")
print("부분단어 기준 (가상):", subword_demo, "→", len(subword_demo), "개")
print("부분단어 예제는 특정 모델의 실제 분할 결과가 아닙니다.")

assert word_tokens == ["The", "unbelievable", "red", "car", "is", "fast", "."]

### 어휘집에서 토큰 ID 찾기

**어휘집**(Vocabulary)은 토크나이저가 구분할 토큰과 해당 ID의 목록이다. 일반적으로 학습할 텍스트를 모아 토큰 후보를 찾고, 등장 빈도나 부분단어 구성 규칙 등으로 항목을 선정해 만든다.

여기서는 **어휘집이 10000종류**라고 가정하지만, 코드에는 예문에 필요한 토큰과 `[PAD]`, `[UNK]` 등의 **일부 항목만 정의**한다. ID 10~16은 실제 언어 모델과 무관한 수업용 번호이며, 모든 새 문장에 대해 동일한 ID 규칙을 적용한다고 가정한다.


In [ ]:
# 설명용 어휘집의 일부 항목만 정의한 딕셔너리
# 실제로 10,000개 항목을 모두 만드는 것은 이 예제에 필요하지 않다.
toy_vocab = {
    "[PAD]": 0, "[UNK]": 1,
    "The": 10, "unbelievable": 11, "red": 12, "car": 13,
    "is": 14, "fast": 15, ".": 16,
}
toy_ids = np.array([toy_vocab.get(token, 1) for token in word_tokens], dtype=np.int64)

for token, token_id in zip(word_tokens, toy_ids):
    print(f"{token:>15s} → ID {token_id}")
print("문장의 ID 배열:", toy_ids.tolist())
assert toy_ids.tolist() == [10, 11, 12, 13, 14, 15, 16]

### 임베딩 테이블에서 필요한 행 조회하기

임베딩 테이블은 토큰 ID별로 **실수 256개로 구성된 벡터**를 저장한 표다. 토큰 종류가 10000개이고 벡터 차원이 256이면 모양은 `(10000, 256)`이다. 공통 문장은 ID가 7개이므로 **일곱 행을 꺼낸 결과**는 `(7, 256)`이다.

아래 코드는 이해를 돕기 위해 임베딩 테이블에 **무작위 초기값**을 넣는다. 모델이 단어의 의미를 이미 배웠다는 뜻이 아니다. 가중치는 **실제 모델 학습 과정에서 조정**되며, 뒤에서는 학습이 끝난 번역 모델의 가중치를 확인한다.


In [ ]:
VOCAB_SIZE = 10_000
EMBED_DIM = 256
rng_emb = np.random.default_rng(42)

# 10,000행 × 256열짜리 교육용 임베딩 표를 무작위 초기화
embedding_table = rng_emb.normal(
    loc=0.0, scale=0.08, size=(VOCAB_SIZE, EMBED_DIM)
).astype("float32")

# 입력 문장의 토큰 ID 7개에 해당하는 행만 조회
embedding_vectors = embedding_table[toy_ids]

print("전체 임베딩 테이블:", embedding_table.shape)
print("문장 토큰 ID:", toy_ids.tolist())
print("문장의 토큰 임베딩:", embedding_vectors.shape)
print("car(ID 13)의 벡터 앞 8개:", np.round(embedding_vectors[3, :8], 3))
print("조회된 car 벡터 = 테이블의 13번 행?:",
      np.array_equal(embedding_vectors[3], embedding_table[13]))

In [ ]:
# 한 행이 한 토큰이고, 가로의 256칸이 그 토큰의 벡터 성분이다.
fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(embedding_vectors, aspect="auto", cmap="coolwarm", vmin=-0.28, vmax=0.28)
ax.set_yticks(range(len(word_tokens)))
ax.set_yticklabels([f"{token} (ID {tid})" for token, tid in zip(word_tokens, toy_ids)])
ax.set_xticks([0, 31, 63, 95, 127, 159, 191, 223, 255])
ax.set_xticklabels(["1", "32", "64", "96", "128", "160", "192", "224", "256"])
ax.set_xlabel("Embedding vector components (256 values)")
ax.set_title("Seven token vectors selected from a 10,000 × 256 table")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
plt.show()

**그림 읽기:** 각 행이 `The`부터 `.`까지의 토큰 하나를 나타내고, 각 열은 256차원 벡터의 한 성분이다. 색은 숫자의 크기를 나타낼 뿐, **현재 무작위 초기값으로 언어적 의미를 해석할 수는 없다.** 256개 숫자를 직접 계산할 필요는 없다.

### 임베딩 테이블이 학습된다는 것은?

모델이 처음 만들어질 때는 많은 가중치가 초기화된다. 이후 학습에서는 예측 결과가 정답과 얼마나 다른지 확인하고, 오차가 줄어들도록 **임베딩을 포함한 신경망의 가중치를 수정**한다.

예를 들어 다음 토큰 예측을 학습하는 모델은 공통 문장의 앞부분 `The unbelievable red`를 보고 다음 토큰 `car`를 맞히는 연습을 할 수 있다. 이런 연습이 수많은 문장에 반복되면 임베딩 테이블의 값도 달라진다.

**여기서는 훈련을 실행하지 않는다.** 앞의 그림은 테이블의 구조를 보여주고, 실제 학습된 숫자는 뒤에서 저장된 번역 모델을 불러와 확인한다.


### 문장 길이와 패딩

입력 토큰 수는 문장마다 달라질 수 있다. 공통 문장 전체는 7개 토큰으로 `(7, 256)`이고, 그 앞부분 `The unbelievable red car`는 4개 토큰으로 `(4, 256)`이다. 문장 길이를 미리 모두 알 필요는 없다.

서로 다른 길이의 입력을 **같은 크기의 배열로 묶는 경우**, 짧은 입력에 **패딩**(Padding)을 붙일 수 있다. 아래는 같은 문장의 7개·4개 토큰을 하나의 배치로 묶어 길이 7에 맞추는 예제다. **모델의 최대 허용 길이가 7이라는 뜻은 아니다.**


In [ ]:
ids_full = toy_ids                          # 7개 토큰
ids_prefix = toy_ids[:4]                    # 같은 문장의 앞부분 4개
PAD_ID = toy_vocab["[PAD]"]                 # ID 0

# 이 배치에서 가장 긴 길이인 7에 맞춤 (모델의 최대 입력 길이와는 다름)
batch_length = max(len(ids_full), len(ids_prefix))
ids_batch = np.array([
    np.pad(ids_full, (0, batch_length - len(ids_full)), constant_values=PAD_ID),
    np.pad(ids_prefix, (0, batch_length - len(ids_prefix)), constant_values=PAD_ID),
], dtype=np.int64)

# 패딩은 실제 토큰이 아님을 알려 주는 마스크: 1=실제, 0=패딩
padding_attention_mask = (ids_batch != PAD_ID).astype(np.int64)
batch_embeddings = embedding_table[ids_batch]

print("배치에 맞춘 길이:", batch_length)
print("패딩된 토큰 ID 배열 (2, 7):\n", ids_batch)
print("패딩 위치를 구별하는 마스크:\n", padding_attention_mask)
print("배치 임베딩 크기:", batch_embeddings.shape)

패딩 토큰은 원래 문장의 정보가 아니므로 실제 토큰과 구별해야 한다. **패딩 마스크**(Padding Mask)는 각 위치가 실제 토큰인지(1), 패딩인지(0) 표시한다. 패딩과 패딩 마스크는 **서로 다른 역할**을 수행한다.

배치에 맞춘 길이는 모델 자체의 **최대 입력 길이**와 별개다. 최대 512토큰을 허용하는 모델이라고 해서 7개 토큰을 반드시 512개로 채워야 하는 것은 아니다.

### 위치 인코딩: 토큰의 순서를 숫자로 표현하기

임베딩 테이블에서 조회한 벡터만으로는 토큰이 문장의 **몇 번째 위치인지** 알 수 없다. 원래 Transformer는 위치 번호 `p=0, 1, ..., 6`에서 **사인·코사인 값을 계산한 256차원 위치 벡터**를 사용한다.

**$d_{\mathrm{model}}$**은 Transformer 내부에서 한 토큰을 나타내는 벡터의 길이다. 여기서는 256으로 정했다. 원래 방식의 위치 벡터는 **학습하는 가중치가 아니라 정해진 수식으로 계산한 값**이다.

앞의 성분은 $\sin(p)$·$\cos(p)$, 다른 성분은 $\sin(p/10)$·$\cos(p/10)$, 또 다른 성분은 $\sin(p/100)$·$\cos(p/100)$처럼 **위치가 바뀔 때 숫자가 변하는 속도**를 달리한다. 이제 코드에서 실제 숫자들을 비교하자.


In [ ]:
# 원래 Transformer의 256차원 사인·코사인 위치 벡터
positions = np.arange(len(word_tokens))[:, None]  # 위치 0~6, (7, 1)
even_dims = np.arange(0, EMBED_DIM, 2)[None, :]
angles = positions / (10000.0 ** (even_dims / EMBED_DIM))

position_encoding = np.zeros((len(word_tokens), EMBED_DIM), dtype=np.float32)
position_encoding[:, 0::2] = np.sin(angles)
position_encoding[:, 1::2] = np.cos(angles)

# 원래 Transformer: 토큰 임베딩을 sqrt(d_model)배 한 뒤 위치 벡터와 더함
d_model = EMBED_DIM
transformer_inputs = embedding_vectors * np.sqrt(d_model) + position_encoding

print("토큰 임베딩:", embedding_vectors.shape)
print("위치 벡터:", position_encoding.shape)
print("Transformer 입력:", transformer_inputs.shape)
print("\n위치 | sin(p) | cos(p) | sin(p/10) | cos(p/10) | sin(p/100)")
for p in (0, 1, 3, 6):
    print(f"{p:4d} | {position_encoding[p, 0]:+.3f} | {position_encoding[p, 1]:+.3f}"
          f" | {position_encoding[p, 64]:+.3f} | {position_encoding[p, 65]:+.3f}"
          f" | {position_encoding[p, 128]:+.3f}")

assert transformer_inputs.shape == (7, 256)
assert np.array_equal(position_encoding[0], np.tile([0, 1], 128))


In [ ]:
# 위치 번호에 따른 서로 다른 변화 속도를 시각화한다.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
xs = np.arange(len(word_tokens))
axes[0].plot(xs, position_encoding[:, 0], "o-", label="sin(p), component 1")
axes[0].plot(xs, position_encoding[:, 64], "s-", label="sin(p/10), component 65")
axes[0].plot(xs, position_encoding[:, 128], "^-", label="sin(p/100), component 129")
axes[0].set_xticks(xs, word_tokens, rotation=35, ha="right")
axes[0].set_ylim(-1.1, 1.1)
axes[0].set_ylabel("Position vector value")
axes[0].set_title("Different rates of position encoding")
axes[0].legend(fontsize=8)

im = axes[1].imshow(position_encoding[:, :128], aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_yticks(xs, word_tokens)
axes[1].set_xticks([0, 31, 63, 95, 127], ["1", "32", "64", "96", "128"])
axes[1].set_title("First 128 of 256 position-vector components")
axes[1].set_xlabel("Position-vector component")
fig.colorbar(im, ax=axes[1], fraction=0.045, pad=0.03)
plt.tight_layout()
plt.show()


**그래프 읽기:** 세 곡선은 같은 위치 번호를 사용하지만 `p`, `p/10`, `p/100`을 입력하므로 **다른 속도로 변화**한다. 오른쪽은 256개 성분 중 앞의 128개를 색상으로 표시한 것이다. 원래 Transformer의 위치 벡터는 특정 단어의 의미가 아니라 **위치 번호에 의해 결정**된다.

### 이미 학습된 번역 모델의 토크나이저와 임베딩

지금까지는 무작위 값으로 만든 임베딩 표를 사용했다. 이제 다른 사람이 **학습을 마친 번역 모델**을 불러와 저장된 토큰화 규칙과 가중치를 살펴본다.

**사전훈련**(Pretraining)은 일반적으로 다른 과제에 적용하기 전에 많은 자료로 기본적인 패턴을 익히는 학습 단계다. 생성형 언어 모델에서는 다음 토큰 예측이 한 사례다. 반면 실제 번역 모델은 **번역 문장 쌍으로 학습**되는 등 학습 목적이 다를 수 있다. 여기서 중요한 점은 **이미 학습된 가중치를 불러와 사용하며 새로 훈련하지 않는다는 사실**이다.

아래에서는 같은 예문을 **모델이 제공하는 실제 토크나이저**로 나눈다. 따라서 앞의 교육용 ID 10~16과는 다른 ID가 나올 수 있다.


In [ ]:
# 영어→스페인어 번역 모델에 포함된 실제 토크나이저
MT_ID = "Helsinki-NLP/opus-mt-en-es"
mt_tokenizer = AutoTokenizer.from_pretrained(MT_ID)
encoded = mt_tokenizer(example_text, return_tensors="pt")

print("같은 원문:", example_text)
print("수업용 단어 토큰 개수:", len(word_tokens))
print("실제 모델의 토큰:", mt_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
print("실제 모델의 토큰 ID:", encoded["input_ids"][0].tolist())
print("실제 입력 ID 배열 모양:", tuple(encoded["input_ids"].shape))
print("토큰에서 텍스트 복원:", mt_tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True))

실제 모델은 자체 어휘집과 토큰 ID를 사용한다. 아래 코드로 번역 모델이 저장한 **임베딩 테이블**에서 이번 문장에 해당하는 행들을 가져와 모양을 확인한다. 이것은 **가중치를 조회하는 작업**이며 학습은 아니다.


In [ ]:
# 번역 모델은 이 절에서 한 번만 불러오고 뒤의 번역 실습에서도 재사용한다.
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_ID)
mt_model.eval()

with torch.inference_mode():
    pretrained_embeddings = mt_model.get_encoder().embed_tokens(encoded["input_ids"])

print("실제 모델의 임베딩 테이블:", tuple(mt_model.get_input_embeddings().weight.shape))
print("실제 모델의 입력 ID 배열:", tuple(encoded["input_ids"].shape))
print("실제 입력에서 조회한 임베딩:", tuple(pretrained_embeddings.shape))
print("첫 토큰 벡터의 앞 8개 값:", pretrained_embeddings[0, 0, :8].tolist())

**정리:** 교육용 임베딩 표는 구조를 관찰하기 위해 무작위 값으로 초기화했다. 실제 번역 모델에 저장된 임베딩은 **학습 과정에서 이미 조정된 가중치**다. 두 경우 모두 토큰 ID로 필요한 행을 조회하지만 **어휘집 크기와 임베딩 차원은 모델에 따라 다르다.**

**확인 질문:** 어휘집을 구축하는 과정을 설명할 수 있는가? 임베딩 테이블 모양 `(10000, 256)`과 문장 입력 모양 `(7, 256)`의 차이는 무엇인가? 위치 벡터 `sin(p)`와 `sin(p/10)`의 변화는 어떻게 다른가? 패딩과 모델의 최대 길이는 왜 구별해야 하는가?


---
## Self-Attention과 Q·K·V: 같은 문장으로 관계 살펴보기

앞에서 사용한 동일한 **7개 토큰**, 즉 `The unbelievable red car is fast.`를 계속 사용한다.

- Q(Query): 현재 `car`가 **어떤 정보를 찾는가**
- K(Key): 각 토큰이 **어떤 정보를 제공할 수 있는가**
- V(Value): 관련성이 계산된 뒤 **실제로 더할 정보**

**순서:** Q·K 관련성 → Softmax로 가중치 → V의 가중합. 이번에도 값을 **교육용으로 미리 설정**하며 모델은 학습하지 않는다.


### 먼저 2차원 숫자로 원리 확인

7개 중 `red`, `car`, `fast`라는 **세 토큰만 잠깐 골라** `car`가 어떤 토큰을 참고하는지 계산해 보자. 이것은 **같은 원문에서 세 개만 뽑은 계산 예제**로, 다른 문장을 새로 도입한 것이 아니다.


In [ ]:
# 세 토큰은 모두 공통 문장에 등장한다.
chosen=['red','car','fast']
query=np.array([1.0,0.0])
keys=np.array([[2.0,0.0],[0.5,0.5],[1.8,0.2]])
values=np.array([[1.0,0.0],[0.5,0.5],[0.0,1.0]])

scores=keys@query/np.sqrt(2)
z=np.exp(scores-scores.max())
weights=z/z.sum()
context=weights@values
for name,wt in zip(chosen,weights):
    print(f'{name:>5}: {wt:.3f}')
print('가중치의 합:',round(float(weights.sum()),3))
print('V의 가중합(2차원 문맥 벡터):',context.round(3))


In [ ]:
fig,ax=plt.subplots(figsize=(6,3))
ax.bar(chosen,weights)
ax.set_ylim(0,1)
ax.set_ylabel('Attention weight')
ax.set_title('Toy Q/K scores for the token "car" (not learned)')
plt.show()


**막대그래프 해석:** `car`의 Q에 대해 `red`와 `fast`의 K가 얼마나 관련되는지에 따라 가중치가 달라진다. 이 가중치를 **각 토큰의 V**에 곱해 합산한다. 숫자는 원리를 위한 예제이고, **언어 의미가 학습된 결과가 아니다.**

### 같은 일곱 토큰을 256차원으로 확장

앞에서 사용한 **수업용 ID 10~16**에 대응하는 **7개 토큰**을 사용한다. 처음에는 256차원 배열을 만들지만, 쉽게 확인하도록 **몇 개 좌표에만 교육용 특징값**을 설정한다. `car`가 `red`와 `fast`를 상대적으로 많이 참고하도록 Q와 K를 고른다.


In [ ]:
attn_tokens=word_tokens                         # 7개 토큰(공통 예문)
d_model=256
X=np.zeros((len(attn_tokens),d_model),dtype=np.float32)
X[2,2]=1.0  # red
X[3,3]=1.0  # car
X[5,5]=1.0  # fast

# 실제로는 학습된 선형 변환들이지만, 여기서는 설명용 값으로 직접 설정.
W_Q=np.zeros((d_model,d_model),dtype=np.float32)
W_K=np.zeros((d_model,d_model),dtype=np.float32)
W_V=np.eye(d_model,dtype=np.float32)
W_Q[3,2]=32.0       # car가 red를 참고하도록 설계
W_Q[3,5]=32.0       # car가 fast를 참고하도록 설계
W_Q[3,3]=16.0       # car 자신도 조금 참고
W_K[2,2]=W_K[3,3]=W_K[5,5]=1.0
Q=X@W_Q
K=X@W_K
V=X@W_V
print('수업용 토큰 ID:',toy_ids.tolist())
print('입력 X:',X.shape,'Q,K,V:',Q.shape,K.shape,V.shape)


### 모든 토큰의 Attention 가중치를 7×7로 보기

각 행이 **현재 토큰**(Query)이고 각 열이 **참고되는 토큰**(Key)이다. 행의 합은 1이며, 이를 V `(7, 256)`에 적용하면 문맥 벡터 `(7, 256)`이 된다.


In [ ]:
score_matrix=(Q@K.T)/np.sqrt(d_model)
z=np.exp(score_matrix-score_matrix.max(axis=1,keepdims=True))
attention_weights=z/z.sum(axis=1,keepdims=True)
context_vectors=attention_weights@V

print('Attention 가중치:',attention_weights.shape)
print('V의 가중합 결과:',context_vectors.shape)
print('car 토큰이 참고하는 가중치 (정확한 값):')
for token,weight in zip(attn_tokens,attention_weights[3]):
    print(f'  {token:>13}: {weight:.3f}')
print('가중치 합:',round(float(attention_weights[3].sum()),3))


In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
im=ax.imshow(attention_weights,cmap='Blues',vmin=0,vmax=.4)
ax.set_xticks(range(len(attn_tokens)),attn_tokens,rotation=35,ha='right')
ax.set_yticks(range(len(attn_tokens)),attn_tokens)
ax.set_xlabel('Key: token being referenced')
ax.set_ylabel('Query: current token')
ax.set_title('7 × 7 toy Self-Attention (designed; NOT learned)')
for i in range(len(attn_tokens)):
    for j in range(len(attn_tokens)):
        w=attention_weights[i,j]
        ax.text(j,i,f'{w:.2f}',ha='center',va='center',fontsize=8,
                color=('white' if w>.26 else 'black'))
fig.colorbar(im,ax=ax,fraction=.044,pad=.03)
plt.show()


**히트맵 읽기:** `car` 행에서 `red`, `fast`가 상대적으로 진하다. 그렇게 되도록 **Q·K를 직접 설계**했기 때문이다. 이는 학습된 모델의 설명이나 주의 분포를 보여주는 것이 아니다.

**핵심은 차원:** 토큰 수가 **7개이므로 Attention 표는 `(7,7)`**이지만 각 토큰의 문맥 벡터는 **여전히 256차원**이므로 출력은 `(7, 256)`이다.

### 다음 절의 Encoder–Decoder 구조와 연결

- **Encoder Self-Attention:** 입력 전체를 참조할 수 있다.
- **Decoder Masked Self-Attention:** 아직 생성하지 않은 미래 토큰을 가린다(**Causal Mask**).
- **Cross-Attention:** Decoder의 Q가 Encoder 출력에서 계산한 K·V를 참고한다.

**Padding Mask는 패딩을 무시하는 것**, **Causal Mask는 미래 출력을 가리는 것**이다. 원래 Transformer의 Decoder는 이를 필요한 방식으로 결합한다.


---
## Encoder–Decoder 활용: 영어 → 스페인어 번역

**번역 모델을 직접 훈련하지 않는다.** 사전훈련된 Encoder–Decoder 계열의 Transformer를 가져와 새 영어 문장들을 번역한다.

원리: **영어 입력 → Encoder 문맥 표현 → Decoder가 다음 스페인어 토큰을 반복 생성**

이전에 불러온 `mt_tokenizer`와 `mt_model`을 재사용한다.


In [ ]:
# 영어→스페인어 번역 예제: 공통 문장을 포함하여 비교
english_sentences = [
    "How is the weather today?",
    "You are welcome.",
    "The unbelievable red car is fast.",
    "I like reading books.",
]

inputs = mt_tokenizer(
    english_sentences,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

# 모델 가중치를 바꾸지 않고 번역 결과만 생성
with torch.inference_mode():
    output_ids = mt_model.generate(**inputs, max_new_tokens=60)

spanish_sentences = mt_tokenizer.batch_decode(
    output_ids, skip_special_tokens=True
)

for en, es in zip(english_sentences, spanish_sentences):
    print("EN:", en)
    print("ES:", es)
    print("-" * 44)


### 관찰 및 해석

1. 공통 문장의 `red`와 `car`의 위치가 스페인어에서 그대로 유지되는가?
2. `You are welcome.`의 번역은 하나만 가능한가?
3. 번역 모델이 영어 단어와 스페인어 단어를 **같은 위치끼리 대응**시키는 방식이라고 생각하는가?

**추가 실험:** `english_sentences` 목록의 문장을 한두 개 바꾼 뒤 같은 셀을 다시 실행해 보자. 훈련이나 설정 변경은 필요하지 않다.


### 세 가지 Transformer 구조의 현재 활용

| 구조 | 대표 활용 | 이 실습에서 |
|---|---|---|
| **Encoder-only** (BERT 등) | 텍스트 분류·의미 검색·임베딩 | **개념만 소개** — 모델 실행 없음 |
| **Encoder–Decoder** (MarianMT 등) | 번역·요약·음성 인식 | **영어→스페인어 번역** |
| **Decoder-only** (GPT 계열 등) | 글쓰기·질의응답·코드 생성 | **이어쓰기·질의응답** |

예를 들어 Encoder-only 기반 임베딩 모델은 `반려견 관리`와 `강아지 돌보기`처럼 표현이 달라도 의미가 비슷한 문서를 찾는 데 활용할 수 있다. **이 구분은 대표적 용도일 뿐**, Decoder-only 모델도 번역 등을 수행할 수 있다.



### 번역 모델에서 LLM으로 연결하기

방금 사용한 모델은 **Encoder–Decoder Transformer**다. 영어 문장을 Encoder가 처리하고, Decoder는 그 내용을 참고하여 스페인어 번역을 생성했다.

이제 사용할 모델은 **Decoder-only Transformer 기반의 소형 언어 모델**이다. 별도의 Encoder나 Cross-Attention 없이 **프롬프트와 이미 생성한 토큰을 한 줄의 시퀀스로 처리**하며, 다음 토큰을 반복 예측한다.

| 이번 Lab의 모델 | 모델 구조 | 무엇을 확인할까? |
|---|---|---|
| 영어→스페인어 번역 모델 | Encoder–Decoder | 입력 문장을 참고한 번역문 생성 |
| 소형 지시문 응답 언어 모델 | Decoder-only | 프롬프트를 이어 쓰고 질문에 답하는 과정 |

두 모델 모두 **이미 학습된 가중치를 불러와 사용**한다. 여기서 *LLM*이라는 말은 단순히 Decoder라는 블록 자체를 뜻하지 않고, **많은 데이터로 학습된 언어 모델**을 가리킨다. 이번 실습의 소형 모델은 이 원리를 관찰하기 위한 교육용 예제이며, 대형 상용 LLM의 성능을 대표하지 않는다.


---
## Decoder-only: 다음 토큰을 예측하고 문장을 생성하기

이번에는 Encoder가 따로 없는 **Decoder-only** 모델을 사용한다. 입력 프롬프트를 보고 다음 토큰의 확률을 계산한 다음, 토큰을 하나씩 추가한다.

빠른 실습을 위해 **작은 모델**을 사용한다. 따라서 출력이 어색하거나 틀릴 수 있다. 여기서 보는 것은 답변 품질이 아니라 **어떻게 텍스트를 생성하는지**이다.


In [ ]:
LM_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
lm_tokenizer = AutoTokenizer.from_pretrained(LM_ID)
lm_model = AutoModelForCausalLM.from_pretrained(LM_ID)
lm_model.eval()

print("사용하는 Decoder-only 모델:", LM_ID)


### 다음 토큰 후보의 확률 확인

모델에 문장을 넣고 **마지막 위치**에서 다음 토큰의 확률을 확인한다. 예제에서는 가장 가능성이 높다고 예측한 상위 5개 토큰을 살펴본다.

실제 모델의 확률이므로 교재의 가상 예시와 일치하지 않을 수 있다.


In [ ]:
prompt = "The capital of France is"
encoded_lm = lm_tokenizer(prompt, return_tensors="pt")

with torch.inference_mode():
    logits = lm_model(**encoded_lm).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)

values, indices = torch.topk(probs, k=5)
print("입력:", prompt)
print("다음 토큰 후보:")
for token_id, prob in zip(indices.tolist(), values.tolist()):
    token = lm_tokenizer.decode([token_id])
    print(f"  {token!r:<18} 확률: {prob:.3f}")


### 활용: 텍스트 이어쓰기와 간단한 질의응답

모델에는 텍스트를 이어 쓰라는 **지시문** 또는 질문을 넣을 수 있다. 두 작업 모두 내부적으로는 다음 토큰 생성이 반복된다.

아래 함수는 '질문/지시문 → 생성 결과'를 간단히 확인하기 위한 것이다. **Chat Template**은 역할(user/assistant)을 모델이 이해하는 입력 형식으로 바꿔준다. 내부 상세 문법은 알 필요가 없다.


In [ ]:
def ask_model(user_message, max_new_tokens=60):
    # 학생은 질문만 바꾼다. 아래 부분은 사전훈련 모델 추론용 공통 코드다.
    messages = [{"role": "user", "content": user_message}]
    token_ids = lm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )

    with torch.inference_mode():
        generated = lm_model.generate(
            token_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # 매번 가장 확률이 높은 토큰 선택 (Greedy)
            pad_token_id=lm_tokenizer.eos_token_id,
        )

    # 모델에 입력한 부분을 제외하고 새로 생성한 부분만 보여준다.
    new_tokens = generated[0, token_ids.shape[-1]:]
    return lm_tokenizer.decode(new_tokens, skip_special_tokens=True)

# 활용 ① 텍스트 이어쓰기
story_prompt = "Continue in one simple sentence: A little dog found a red ball."
print("[이어쓰기]")
print(ask_model(story_prompt, max_new_tokens=45))

# 활용 ② 간단한 질의응답
question = "What is the capital of Spain? Answer in one short sentence."
print()
print("[질의응답]")
print(ask_model(question, max_new_tokens=40))


**예제에서 확인할 점:** 이 모델이 답을 맞히거나 말을 자연스럽게 한다는 사실보다, 두 작업을 **같은 Decoder-only 모델**로 수행했다는 사실이 핵심이다. 아주 작은 교육용 모델이므로 정답을 틀리거나 의미 없는 문장을 생성할 수 있다.


---
## P4 Project 미리보기 — 의미 검색과 답변 생성 연결하기

이번 Lab에서는 **번역**(Encoder–Decoder)과 **텍스트 생성**(Decoder-only)을 각각 실행했다. Project에서는 아직 실행하지 않은 **Encoder-only 계열의 문장 임베딩 모델**을 추가하여 두 기능을 연결한다.

**프로젝트 질문 예시:** `같은 동물을 여러 번 촬영했을 때 개체 수를 어떻게 세나요?`

| 단계 | 사용하는 모델 또는 자료 | 학생이 관찰할 내용 |
|---|---|---|
| ① 문서 준비 | 가상 한솔 생태관측소 안내문 **25개** | 문서마다 서로 다른 정보를 담고 있음 |
| ② 의미 검색 | **Encoder 계열** 문장 임베딩 모델 | 단어가 정확히 같지 않아도 관련 문서 검색 |
| ③ 답변 생성 | **Decoder-only** 언어 모델 | 검색된 문서를 읽고 질문에 대한 문장 생성 |
| ④ 결과 검토 | 학생이 원문과 비교 | 답변에 근거가 있는지, 잘못 만든 내용이 있는지 판단 |

```text
질문 → 의미가 유사한 문서 3개 찾기 → 문서를 함께 전달 → 답변 생성 → 근거 확인
       (Encoder 계열)                 (Decoder-only)
```

**Lab과 Project의 차이:** 이 Lab에서는 이미 준비된 모델의 입력과 출력을 *관찰*했다. Project에서는 **단순 키워드 검색과 의미 검색을 비교하고**, 검색 결과를 생성 모델에 제공하여 **답변의 근거 적합성**까지 평가한다.

**필요한 파일:** `P4_project_document_search_RAG.ipynb`, `P4_project_documents_25.json` (같은 폴더에 보관). Project에서도 **모델 훈련은 없다.**

## 실습 마무리 — 결과를 근거로 설명하기

다음 질문에 코드 출력을 보고 **각각 1~2문장**으로 답해 보자.

1. 토큰과 토큰 ID, 임베딩 벡터는 무엇이 다른가?
2. Attention 예제에서 Q·K와 V의 역할은 무엇이었나?
3. Encoder–Decoder 번역에서는 Encoder와 Decoder가 각각 무엇을 했는가?
4. Decoder-only 모델은 왜 영어 프롬프트를 입력받아도 Encoder가 필요 없는가?
5. 번역과 질의응답 모두 **다음 토큰 생성**이라는 공통 원리를 공유한다는 설명은 어떤 뜻인가?
6. Decoder-only **구조**와 이를 사전훈련한 **언어 모델**을 어떻게 구분할 수 있을까?
7. Project에서는 Encoder 계열과 Decoder-only 모델이 각각 어떤 역할을 하는가?

**범위 확인:** 이 노트북에는 학습 루프·손실함수 최적화·파인튜닝·강화학습 코드가 없다. 전체 코드는 **사전훈련된 모델의 추론 또는 숫자 예제**이다.

### 모델 출처

- [Helsinki-NLP/opus-mt-en-es](https://huggingface.co/Helsinki-NLP/opus-mt-en-es)
- [HuggingFaceTB/SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)

**다음 학습:** `P4_project_document_search_RAG.ipynb` — 키워드 검색·의미 검색·근거 기반 답변 생성 비교.
